# ADA Noble · Complex state space recurrence

Separate the historical Noble recurrence from the exponential-trapezoidal
equation it cites. This notebook ports the source scan to PyTorch, checks
both recurrences independently, and trains a small causal token predictor
using the paper-aligned equation.

**Sources.** Historical JAX/Flax source: Ada-noble/ada_noble_1.py, SHA-256 d57ba9e4e0c10a5dd7b27e57bafedff412bbb8f651467559bc07a6df940d0020.
Its Mamba3Block is the specimen. The workspace's larger exploratory Noble
notebook contains other architecture sketches. This notebook focuses on
the reproducible recurrence in the named source file.
The [Mamba-3 paper](https://arxiv.org/html/2603.15569v1) supplies
Proposition 1 and the matrix-state comparison in Section 3.3.

This is a mathematical CPU port and a small model experiment. It does
not execute the original Flax model, recreate its entire GQA/MoE stack,
or verify accelerator kernels, scaling, and performance claims.

### Run environment

Use Python 3.11 or 3.12 with Jupyter and a CPU PyTorch build (2.6 or later).
Select that environment as the kernel and choose **Restart Kernel and Run All**.
For a fresh environment, install Python packages with
**python -m pip install jupyterlab torch**, selecting the
[CPU wheels](https://pytorch.org/get-started/locally/) if desired.
The cells use synthetic tensors and make no network requests.

In [1]:
from io import BytesIO
import math
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple
from types import SimpleNamespace

import torch
from torch import Tensor, nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(2026)
torch.use_deterministic_algorithms(True)
print("PyTorch", torch.__version__, "| device cpu | seed 2026")

PyTorch 2.11.0+cpu | device cpu | seed 2026


## Historical scan port

The Flax source constructs a real negative decay and optional complex
B/C features. It compresses each group of input channels to a sum.
At each position it decays the state and averages adjacent input
contributions scaled by the current position's zero-order-hold factor.
The stored previous timestep does not influence the original update.
A complex-valued state here still uses a real transition. This
does not implement a learned rotational transition.

In [2]:
def historical_noble_scan(group_x, delta, real_decay, b, c):
    if real_decay.is_complex():
        raise TypeError("historical Noble decay is real")
    state = torch.zeros_like(b[:, 0])
    previous_input = torch.zeros_like(state)
    outputs = []
    for x_t, dt_t, b_t, c_t in zip(
        group_x.unbind(1), delta.unbind(1), b.unbind(1), c.unbind(1), strict=True
    ):
        dt_t = dt_t.unsqueeze(-1)
        transition = (dt_t * real_decay).exp()
        factor = torch.where(
            real_decay.abs() < 1e-6, dt_t, (transition - 1.0) / (real_decay + 1e-8),
        )
        current_input = b_t * x_t.sum(-1, keepdim=True)
        state = transition * state + 0.5 * factor * (current_input + previous_input)
        outputs.append((c_t.conj() * state).sum(-1).real)
        previous_input = current_input
    return torch.stack(outputs, dim=1)

## Paper-aligned diagonal reference

Let u_t = B_t x_t. Proposition 1 uses
alpha_t = exp(delta_t A_t), beta_t = (1 - lambda_t) delta_t alpha_t,
and gamma_t = lambda_t delta_t. Update the state as
h_t = alpha_t h_(t-1) + beta_t u_(t-1) + gamma_t u_t.
We use a time-independent diagonal complex A for this fixture and read
out the real part of the conjugate-dot product with C_t. The implementation
returns both h_t and u_t at a chunk boundary. Both are needed to resume.

In [3]:
def trapezoid_scan(inputs, delta, mix, decay, carry=None):
    if not inputs.shape[1]:
        raise ValueError("a chunk must contain at least one position")
    if carry is None:
        state = torch.zeros_like(inputs[:, 0])
        previous = torch.zeros_like(state)
    else:
        state, previous = carry
    history = []
    for source, dt_t, mix_t in zip(
        inputs.unbind(1), delta.unbind(1), mix.unbind(1), strict=True
    ):
        dt_t, mix_t = dt_t.unsqueeze(-1), mix_t.unsqueeze(-1)
        alpha = (dt_t * decay).exp()
        state = (
            alpha * state + (1 - mix_t) * dt_t * alpha * previous
            + mix_t * dt_t * source
        )
        history.append(state)
        previous = source
    return torch.stack(history, dim=1), (state, previous)


def dense_state_reference(inputs, delta, mix, decay):
    alpha = (delta[..., None] * decay).exp()
    beta = (1 - mix[..., None]) * delta[..., None] * alpha
    gamma = mix[..., None] * delta[..., None]
    states = []
    for t in range(inputs.shape[1]):
        row = []
        for j in range(t + 1):
            coefficient = gamma[:, j] * alpha[:, j + 1:t + 1].prod(dim=1)
            if j < t:
                coefficient = coefficient + beta[:, j + 1] * alpha[:, j + 2:t + 1].prod(dim=1)
            row.append(coefficient * inputs[:, j])
        states.append(torch.stack(row, dim=1).sum(dim=1))
    return torch.stack(states, dim=1)

## Recurrence results

Check four independent properties. A two-step scalar fixture verifies
the historical equation by ordinary arithmetic. A dense unrolled
construction checks the reference scan in complex128. A two-chunk run
verifies state and previous input carry. Finally compare the two
formulations under the same real decay and lambda = 0.5. A match
between them would be unexpected under the chosen varying timesteps.

In [4]:
probe_x = torch.tensor([[[[1.0, 2.0]], [[0.0, 0.0]]]], dtype=torch.float64)
probe_dt = torch.tensor([[[0.2], [0.5]]], dtype=torch.float64)
probe_a = torch.tensor([[-0.4]], dtype=torch.float64)
probe_b = torch.ones(1, 2, 1, 1, dtype=torch.complex128)
probe_c = torch.ones_like(probe_b)
probe_output = historical_noble_scan(probe_x, probe_dt, probe_a, probe_b, probe_c)
alpha_0, alpha_1 = math.exp(-0.08), math.exp(-0.2)
old_first = 1.5 * (alpha_0 - 1.0) / (-0.4 + 1e-8)
old_second = alpha_1 * old_first + 1.5 * (alpha_1 - 1.0) / (-0.4 + 1e-8)
torch.testing.assert_close(
    probe_output.flatten(), torch.tensor([old_first, old_second], dtype=torch.float64),
    atol=1e-12, rtol=1e-12,
)
print("Historical Noble two-step arithmetic: PASS")

dt = 0.05 + 0.9 * torch.rand(2, 7, 3, dtype=torch.float64)
blend = 0.1 + 0.8 * torch.rand_like(dt)
rates = torch.complex(-torch.rand(3, 4, dtype=torch.float64), torch.rand(3, 4, dtype=torch.float64))
source_inputs = torch.randn(2, 7, 3, 4, dtype=torch.complex128)
whole, whole_carry = trapezoid_scan(source_inputs, dt, blend, rates)
dense = dense_state_reference(source_inputs, dt, blend, rates)
torch.testing.assert_close(whole, dense, atol=1e-12, rtol=1e-12)
first, carry = trapezoid_scan(source_inputs[:, :3], dt[:, :3], blend[:, :3], rates)
second, split_carry = trapezoid_scan(source_inputs[:, 3:], dt[:, 3:], blend[:, 3:], rates, carry)
torch.testing.assert_close(whole, torch.cat((first, second), dim=1), atol=0, rtol=0)
for lhs, rhs in zip(whole_carry, split_carry, strict=True):
    torch.testing.assert_close(lhs, rhs, atol=0, rtol=0)
print(f"Complex scan/dense max delta: {(whole - dense).abs().max():.3e}")
print("Full/chunk equivalence including previous input: PASS")

probe_u = probe_b * probe_x.sum(-1, keepdim=True)
fixed_blend = torch.full_like(probe_dt, 0.5)
paper, _ = trapezoid_scan(probe_u, probe_dt, fixed_blend, probe_a)
mismatch = (probe_output - paper.real.sum(-1)).abs().max().item()
assert mismatch > 1e-3
print(f"Historical vs paper recurrence difference: {mismatch:.8f} | source parity FAIL (expected)")

unit_blend = torch.ones_like(dt)
euler, _ = trapezoid_scan(source_inputs, dt, unit_blend, rates)
h = torch.zeros_like(source_inputs[:, 0])
for t in range(source_inputs.shape[1]):
    h = torch.exp(dt[:, t, :, None] * rates) * h + dt[:, t, :, None] * source_inputs[:, t]
    torch.testing.assert_close(h, euler[:, t], rtol=1e-12, atol=1e-12)
print("lambda = 1 exponential-Euler limit: PASS")

Historical Noble two-step arithmetic: PASS
Complex scan/dense max delta: 7.022e-16
Full/chunk equivalence including previous input: PASS
Historical vs paper recurrence difference: 0.05614370 | source parity FAIL (expected)
lambda = 1 exponential-Euler limit: PASS


## Why group sums are different from matrix-state MIMO

A diagnostic uses Section 3.3's one-step exponential-Euler example.
With H shaped (N, P), B and C shaped (N, R), and X shaped (P, R),
update with B @ X.T and read with H.T @ C. Swapping channels with
equal grouped sums can change matrix-state output. A grouped sum
cannot make that distinction. The following is a shape and behavior
check, not a training or hardware implementation of the paper's MIMO.

In [5]:
def matrix_state_step(state, x, b, c, alpha, delta):
    state = alpha * state + delta * (b @ x.transpose(-1, -2))
    return state, state.transpose(-1, -2) @ c


zero = torch.zeros(2, 2)
x = torch.tensor([[2.0, 0.0], [0.0, 1.0]])
swapped = x.flip(-1)
b = torch.tensor([[2.0, 0.0], [0.0, 1.0]])
c = torch.eye(2)
torch.testing.assert_close(x.sum(dim=-1), swapped.sum(dim=-1))
_, out_x = matrix_state_step(zero, x, b, c, 0.8, 0.5)
_, out_swapped = matrix_state_step(zero, swapped, b, c, 0.8, 0.5)
assert out_x.shape == out_swapped.shape == (2, 2)
assert not torch.allclose(out_x, out_swapped)
print("Equal pooled inputs, distinct matrix-state readouts: PASS")

Equal pooled inputs, distinct matrix-state readouts: PASS


## Train and evaluate a tiny Noble-inspired model

Reuse the historical block's high-level structure of causal depthwise
convolution, projected timesteps, grouped channels, a skip path and
gated readout. The recurrence below uses the paper-aligned coefficients
and a learned diagonal complex transition per group. It compresses
input groups as in Noble. Full MIMO, GQA and MoE are outside this run.

Train on synthetic modular sequences with disjoint held-out start IDs.
Check gradient flow through both real and imaginary transition parameters
and guard prefix invariance before drawing conclusions about fitting.

In [6]:
def cyclic_examples(starts, length, vocab):
    return (torch.as_tensor(starts)[:, None] + torch.arange(length)[None, :]) % vocab


def next_token_metrics(logits, ids):
    prediction = logits[:, :-1]
    labels = ids[:, 1:]
    loss = F.cross_entropy(prediction.flatten(0, 1), labels.flatten())
    accuracy = (prediction.argmax(-1) == labels).float().mean()
    return loss, accuracy

In [7]:
class TinyNobleMixer(nn.Module):
    def __init__(self, width=24, groups=6, states=4):
        super().__init__()
        if width % groups:
            raise ValueError("width must be divisible by groups")
        self.groups = groups
        self.states = states
        self.input_gate = nn.Linear(width, 2 * width)
        self.convolution = nn.Conv1d(width, width, kernel_size=3, groups=width)
        self.dt = nn.Sequential(nn.Linear(width, 6, bias=False), nn.Linear(6, groups))
        self.mix = nn.Linear(width, groups)
        self.bc = nn.Linear(width, groups * states * 4)
        self.decay_log = nn.Parameter(torch.zeros(groups, states))
        self.frequency = nn.Parameter(torch.linspace(0.15, 0.65, states).repeat(groups, 1))
        self.output = nn.Linear(width, width)

    def forward(self, x):
        states, gate = self.input_gate(x).chunk(2, dim=-1)
        states = F.silu(self.convolution(F.pad(states.transpose(1, 2), (2, 0)))).transpose(1, 2)
        batch, length, width = states.shape
        dt = F.softplus(self.dt(states))
        mix = self.mix(states).sigmoid()
        a = torch.complex(-F.softplus(self.decay_log), self.frequency)
        br, bi, cr, ci = self.bc(states).reshape(
            batch, length, self.groups, 4 * self.states
        ).chunk(4, dim=-1)
        grouped = states.reshape(batch, length, self.groups, -1).sum(-1, keepdim=True)
        input_terms = torch.complex(br, bi) * grouped
        trajectory, _ = trapezoid_scan(input_terms, dt, mix, a)
        y = (torch.complex(cr, ci).conj() * trajectory).sum(-1).real
        y = y.repeat_interleave(width // self.groups, dim=-1)
        return x + self.output(y * F.silu(gate))


class TinyNobleLM(nn.Module):
    def __init__(self, vocab=16, width=24):
        super().__init__()
        self.embed = nn.Embedding(vocab, width)
        self.mixer = TinyNobleMixer(width=width)
        self.norm = nn.LayerNorm(width)
        self.head = nn.Linear(width, vocab)

    def forward(self, ids):
        return self.head(self.norm(self.mixer(self.embed(ids))))


torch.manual_seed(2028)
model = TinyNobleLM()
train = cyclic_examples(range(12), 10, 16)
val = cyclic_examples(range(12, 16), 10, 16)
assert torch.unique(torch.cat((train, val)), dim=0).shape[0] == 16
paired = val[:1].clone()
paired[:, 4:] = (paired[:, 4:] + 5) % 16


@torch.no_grad()
def eval_and_validate():
    model.eval()
    logits = model(val)
    torch.testing.assert_close(logits[:1, :4], model(paired)[:, :4], rtol=0, atol=2e-6)
    torch.testing.assert_close(logits[:1, :4], model(val[:1, :4]), rtol=0, atol=2e-6)
    return [item.item() for item in next_token_metrics(logits, val)]


before, initial_accuracy = eval_and_validate()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.02)
model.train()
for step in range(32):
    optimizer.zero_grad(set_to_none=True)
    loss, _ = next_token_metrics(model(train), train)
    assert torch.isfinite(loss)
    loss.backward()
    if step == 0:
        for parameter in (model.mixer.decay_log, model.mixer.frequency):
            assert torch.isfinite(parameter.grad).all() and parameter.grad.abs().sum() > 0
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
after, final_accuracy = eval_and_validate()
assert after < before
print(f"Held-out CE: {before:.6f} -> {after:.6f}")
print(f"Held-out accuracy: {initial_accuracy:.1%} -> {final_accuracy:.1%}")
print("Prefix and truncation invariance before/after training: PASS")

with torch.no_grad():
    ids = torch.tensor([[14, 15]])
    for _ in range(6):
        ids = torch.cat((ids, model(ids)[:, -1].argmax(-1, keepdim=True)), dim=1)
assert ids.shape == (1, 8)
print("Greedy IDs:", ids.tolist())

Held-out CE: 3.306319 -> 0.004582
Held-out accuracy: 0.0% -> 100.0%
Prefix and truncation invariance before/after training: PASS
Greedy IDs: [[14, 15, 0, 1, 2, 3, 4, 5]]


## Decision

Preserve the historical equations and their reproducible discrepancy as a
regression fixture. Candidate next work includes exact JAX/PyTorch
tensor-level parity, parameter matching, long-horizon numerical analysis
and benchmarks on state-tracking tasks. Porting the full Flax stack or
implementing actual MIMO/decode kernels requires separate validation.